# Batch 5C — Rev. 4 Frontier API Baseline

This notebook runs the registered 36-row frontier-model comparison for RQ2. It uses the exact frozen questions, ordered evidence windows, free-form prompt/parser, ASK policy, and offline verifier, while replacing the Qwen answer call with the OpenAI Responses API model alias `gpt-6-astra`.

A GPU is not needed: inference runs through the API. Use a CPU runtime to preserve Colab compute units.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Before running

In Colab's **Secrets** panel (key icon), add a secret named `OPENAI_API_KEY` and enable notebook access. The API project must have billing or credits; ChatGPT and Colab subscriptions do not provide OpenAI API credits. The notebook never prints or saves the key.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = 'https://github.com/Wenjia1215/ComplianceGPT.git'
SOURCE_COMMIT = 'a12aa781e74077432aaedf8a29ccc5d993408e94'
REPO_DIR = Path('/content/ComplianceGPT_batch5c')
OUTPUT_DIR = Path('/content/drive/MyDrive/rq2_frontier_baseline_v1')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if (REPO_DIR / '.git').is_dir():
    dirty = subprocess.check_output(
        ['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True
    ).strip()
    if dirty:
        raise RuntimeError(
            f'{REPO_DIR} contains local changes. Start a fresh runtime or move that directory.'
        )
    subprocess.run(['git', 'fetch', 'origin', 'main'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--filter=blob:none', REPOSITORY_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_COMMIT], cwd=REPO_DIR, check=True)
checked_out = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert checked_out == SOURCE_COMMIT, (checked_out, SOURCE_COMMIT)
print('Pinned Batch 5C source:', checked_out)
print('Persistent result directory:', OUTPUT_DIR)

In [ ]:
import json
from importlib.metadata import version

runtime_manifest = OUTPUT_DIR / 'manifests/api_runtime.json'
packages = ['numpy==2.0.2', 'PyYAML==6.0.3']
if runtime_manifest.is_file():
    prior_runtime = json.loads(runtime_manifest.read_text(encoding='utf-8'))
    prior_openai = str(prior_runtime.get('openai_python', '') or '').strip()
    if not prior_openai:
        raise RuntimeError('Existing API runtime manifest has no OpenAI SDK version.')
    packages.append(f'openai=={prior_openai}')
    print('Resuming with recorded OpenAI SDK:', prior_openai)
else:
    packages.append('openai>=1.99.0,<3')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *packages], check=True)
print('OpenAI SDK:', version('openai'))

In [ ]:
import os
from google.colab import userdata

try:
    api_key = str(userdata.get('OPENAI_API_KEY') or '').strip()
except Exception as exc:
    raise RuntimeError(
        "Add OPENAI_API_KEY in Colab Secrets (key icon), enable notebook access, then rerun this cell."
    ) from exc
if not api_key:
    raise RuntimeError('OPENAI_API_KEY is empty in Colab Secrets.')
os.environ['OPENAI_API_KEY'] = api_key
del api_key
print('OPENAI_API_KEY loaded into process memory; its value was not displayed or written.')

In [ ]:
test_environment = os.environ.copy()
test_environment['PYTHONPATH'] = str(REPO_DIR / 'src')
subprocess.run(
    [
        sys.executable, '-m', 'unittest',
        'tests.test_frontier_api_answerer',
        'tests.test_rq2_frontier_baseline', '-v',
    ],
    cwd=REPO_DIR, env=test_environment, check=True,
)
print('Batch 5C offline tests passed.')

In [ ]:
import csv
import hashlib
from IPython.display import Markdown, display

FROZEN_ARCHIVE_SHA256 = '56a70db6eca0420df6affbe63c859418ac423b54d80d3d1eae7bf785f0f63328'
frozen_archive = REPO_DIR / 'experiments/answerer_comparison/rq2_matched/results_v3/compliancegpt_rq2_matched_v3.zip'
actual_archive_sha256 = hashlib.sha256(frozen_archive.read_bytes()).hexdigest()
assert actual_archive_sha256 == FROZEN_ARCHIVE_SHA256, (
    actual_archive_sha256, FROZEN_ARCHIVE_SHA256
)
runner = REPO_DIR / 'experiments/answerer_comparison/rq2_frontier_baseline/run_frontier_baseline.py'
prepare_command = [
    sys.executable, '-u', str(runner),
    '--repo-root', str(REPO_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--prepare-only',
]
subprocess.run(prepare_command, cwd=REPO_DIR, check=True)
contexts_path = OUTPUT_DIR / 'contexts/rev4_prepared_contexts.jsonl'
with contexts_path.open('r', encoding='utf-8') as handle:
    prepared_rows = sum(1 for line in handle if line.strip())
assert prepared_rows == 36, f'Expected 36 prepared contexts, found {prepared_rows}'
preflight_summary = json.loads((OUTPUT_DIR / 'preflight/summary.json').read_text(encoding='utf-8'))
assert preflight_summary.get('scope') == 'gold-free frozen-context integrity audit; no API call'
assert preflight_summary.get('api_calls_performed') == 0
print('Frozen archive verified:', actual_archive_sha256)
print(f'Prepared-context audit passed: {prepared_rows} Rev. 4 contexts; no API call made.')
display(Markdown((OUTPUT_DIR / 'PREPARED_CONTEXTS.md').read_text(encoding='utf-8')))

## Run the registered API comparison

The first-attempt prompts contain about 227,000 characters in total (roughly 57,000 tokens by a simple character estimate). Actual API token usage is recorded from each response. A parse failure can trigger up to two additional calls for that row.

The cell checkpoints every completed row and every returned API response. After a disconnect, reconnect and use **Runtime → Run all**; validated result rows are skipped.

In [ ]:
from collections import deque

run_command = [
    sys.executable, '-u', str(runner),
    '--repo-root', str(REPO_DIR),
    '--output-dir', str(OUTPUT_DIR),
]
run_environment = os.environ.copy()
tail = deque(maxlen=400)
print('Starting the 36-row Rev. 4 frontier API baseline.')
print('After a disconnect, reconnect and Run all; validated rows are skipped.')
process = subprocess.Popen(
    run_command, cwd=REPO_DIR, env=run_environment,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='')
    tail.append(line)
return_code = process.wait()
failure_log = OUTPUT_DIR / 'runner_failure_tail.log'
if return_code != 0:
    failure_log.write_text(''.join(tail), encoding='utf-8')
    print(f'Batch 5C failed. The final runner output is saved at {failure_log}.')
    raise subprocess.CalledProcessError(return_code, run_command)
if failure_log.exists():
    failure_log.unlink()
print('Batch 5C runner completed successfully.')

In [ ]:
summary_path = OUTPUT_DIR / 'summary.json'
api_manifest_path = OUTPUT_DIR / 'manifests/api_responses.json'
assert summary_path.is_file(), summary_path
assert api_manifest_path.is_file(), api_manifest_path
summary = json.loads(summary_path.read_text(encoding='utf-8'))
api_manifest = json.loads(api_manifest_path.read_text(encoding='utf-8'))
assert summary.get('n_questions') == 36
assert api_manifest.get('completed_result_rows') == 36
display(Markdown((OUTPUT_DIR / 'SUMMARY.md').read_text(encoding='utf-8')))
print('Server-reported model:', api_manifest['response_model'])
print('All logged API usage:', api_manifest['all_logged_usage']['totals'])
print('Parse-retry rows:', api_manifest['parse_retry_rows'])
print('Orphaned billed calls:', api_manifest['orphaned_call_count'])

In [ ]:
archive_path = OUTPUT_DIR.parent / f'{OUTPUT_DIR.name}.zip'
assert archive_path.is_file(), f'Result ZIP was not created: {archive_path}'
archive_sha256 = hashlib.sha256(archive_path.read_bytes()).hexdigest()
print('Confirmed result directory:', OUTPUT_DIR)
print('Confirmed result ZIP:', archive_path)
print('Result ZIP SHA-256:', archive_sha256)
print('Download this executed notebook from Colab and attach it together with the ZIP for independent audit.')

## Handoff

After the final cell succeeds, download this executed `.ipynb` from Colab and provide it together with `/content/drive/MyDrive/rq2_frontier_baseline_v1.zip`. The executed notebook, raw API log, row-level contracts, summary, and hashes are all needed before the result is added to the repository.